In [ ]:
%pip install prophet

In [ ]:
from prophet import Prophet
import pandas as pd

In [ ]:
data = {
    'ds': pd.to_datetime([
        '2023-04-01', '2023-04-02', '2023-04-03', '2023-04-04', '2023-04-05',
        '2023-04-06', '2023-04-07', '2023-04-08', '2023-04-09', '2023-04-10',
        '2023-04-11', '2023-04-12', '2023-04-13', '2023-04-14', '2023-04-15',
        '2023-04-16', '2023-04-17', '2023-04-18', '2023-04-19', '2023-04-20'
    ]),
    'y': [
        120, 90, 85, 88, 92,
        95, 125, 130, 95, 80,
        80, 250, 320, 280, 150, # ช่วงสงกรานต์ที่คุณยกตัวอย่าง
        90, 85, 88, 91, 94
    ]
}
df = pd.DataFrame(data)

# แสดงข้อมูล 5 แถวแรก
print(df.head())

In [ ]:
holidays = pd.DataFrame({
  'holiday': 'songkran',
  'ds': pd.to_datetime(['2023-04-13']),
  'lower_window': -2,  # ให้เริ่มมีผลกระทบ 2 วันก่อนวันจริง (คือวันที่ 11, 12)
  'upper_window': 1,   # และให้มีผลกระทบต่อไปอีก 1 วันหลังวันจริง (คือวันที่ 14)
})

print(holidays)

In [ ]:
model = Prophet(holidays=holidays, weekly_seasonality=True, yearly_seasonality=False)

# สอน Model ด้วยข้อมูลของเรา
model.fit(df)

In [ ]:
future = model.make_future_dataframe(periods=10)

In [ ]:
# พยากรณ์
forecast = model.predict(future)

In [ ]:
# yhat คือค่าพยากรณ์หลัก
# yhat_lower และ yhat_upper คือช่วงความไม่แน่นอน

print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10))

In [ ]:
fig1 = model.plot(forecast)

In [ ]:
fig2 = model.plot_components(forecast)

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
date_rng = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
df = pd.DataFrame(date_rng, columns=['date'])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
np.random.seed(42) # เพื่อให้ผลลัพธ์เหมือนกันทุกครั้ง
noise = np.random.randint(0, 15, size=(len(df)))

In [ ]:
weekday_effect = df['date'].dt.dayofweek.apply(lambda x: 25 if x >= 5 else 0) # 5=เสาร์, 6=อาทิตย์

In [ ]:
time_trend = np.arange(len(df)) * 0.1

In [ ]:
df['patients'] = 80 + weekday_effect + time_trend + noise

In [ ]:
df.set_index('date', inplace=True)

df.head()

In [ ]:
df['patients'].plot(figsize=(15, 5), title='Daily ER Patient Count (Simulated Data)')
plt.xlabel('Date')
plt.ylabel('Number of Patients')
plt.show()

In [ ]:
def create_features(df):
    # สร้าง features จาก date

    df['dayofweek'] = df.index.dayofweek # 0 = จันทร์, 6 = อาทิตย์
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['dayofyear'] = df.index.dayofyear

    # สร้าง Lag Features (ข้อมูลย้อนหลัง)
    # เราจะใช้ข้อมูล 7 วันที่แล้ว และ 14 วันที่แล้วมาช่วยทาย
    df['lag_7'] = df['patients'].shift(7)
    df['lag_14'] = df['patients'].shift(14)

    # สร้าง ค่าเฉลี่ยย้อนหลัง
    df['rolling_mean_7'] = df['patients'].shift(1).rolling(window=7).mean()

    return df

df = create_features(df)

# ข้อมูลจะมีค่าว่าง (NaN) ในช่วงแรกๆ เพราะยังไม่มีข้อมูลย้อนหลังให้คำนวณ
# เราจะลบแถวที่มีค่าว่างทิ้งไป
df.dropna(inplace=True)

df.head()

In [ ]:
FEATURES = ['dayofweek', 'quarter', 'month', 'dayofyear', 'lag_7', 'lag_14', 'rolling_mean_7']
TARGET = 'patients'

In [ ]:
X = df[FEATURES]
y = df[TARGET]

In [ ]:
split_date = '2024-10-01'
X_train, X_test = X.loc[X.index < split_date], X.loc[X.index >= split_date]
y_train, y_test = y.loc[y.index < split_date], y.loc[y.index >= split_date]

In [ ]:
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100,
                             learning_rate=0.05, # ค่อยๆ เรียนรู้ช้าๆ
                             early_stopping_rounds=10, # ถ้าผลไม่ดีขึ้น 10 รอบให้หยุด
                             random_state=42)

In [ ]:
# XGBoost ต้องมี evaluation set เพื่อใช้ early stopping
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

In [ ]:
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

In [ ]:
print(f"\nความคลาดเคลื่อนเฉลี่ยของ Random Forest: {rf_mae:.2f} คน")
print(f"ความคลาดเคลื่อนเฉลี่ยของ XGBoost: {xgb_mae:.2f} คน")

In [ ]:
results = pd.DataFrame({
    'Actual': y_test,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred
})

In [ ]:
results.plot(figsize=(15, 7), style=['-', '--', ':'],
             title='Comparison of Forecast vs Actual Data')
plt.ylabel('Number of Patients')
plt.show()

In [ ]:
def plot_feature_importance(model, features, model_name):
    # สร้าง DataFrame ของ feature importance
    fi = pd.DataFrame(data=model.feature_importances_,
                      index=features,
                      columns=['importance'])
    fi_sorted = fi.sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=fi_sorted.index, y=fi_sorted['importance'])
    plt.title(f'Feature Importance of the {model_name} Model') # Changed title to English
    plt.xticks(rotation=45)
    plt.xlabel('Features') # Added x-axis label
    plt.show()

In [ ]:
plot_feature_importance(xgb_model, FEATURES, 'XGBoost')

In [ ]:
plot_feature_importance(rf_model, FEATURES, 'Random Forest')

In [ ]:
pip install git+https://github.com/amazon-science/chronos-forecasting.git

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from chronos import ChronosPipeline

In [ ]:
pipeline = ChronosPipeline.from_pretrained(
  "amazon/chronos-t5-small",
  device_map="cuda",
  dtype=torch.bfloat16,
)

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/AileenNielsen/TimeSeriesAnalysisWithPython/master/data/AirPassengers.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
context = torch.tensor(df["#Passengers"])
prediction_length = 12
forecast = pipeline.predict(context, prediction_length)

In [ ]:
forecast_index = range(len(df), len(df) + prediction_length)
low, median, high = np.quantile(forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0)

plt.figure(figsize=(8, 4))
plt.plot(df["#Passengers"], color="royalblue", label="historical data")
plt.plot(forecast_index, median, color="tomato", label="median forecast")
plt.fill_between(forecast_index, low, high, color="tomato", alpha=0.3, label="80% prediction interval")
plt.legend()
plt.grid()
plt.show()

In [ ]:
forecast[0][:][0]

In [ ]:
# คำนวณค่ามัธยฐานของผลการพยากรณ์ในแต่ละช่วงเวลา
median_forecast = np.median(forecast[0], axis=0)

print("ค่ามัธยฐานของผลการพยากรณ์ 12 เดือนข้างหน้า")
print(median_forecast)